# SignalBridge: Selecting K for Financial Company Archetypes — V2

**Purpose:** identify interpretable financial archetypes from each company's
latest available financial snapshot. The resulting financial clusters will
become structured signals for the later company-ranking system.

V2 addresses the failure observed in V1:

- K=2 is excluded because two groups cannot support the intended multi-archetype framework.
- Sparse, unbounded ratios no longer drive the clustering distance.
- Core clustering uses higher-coverage financial variables.
- Sparse cash, fixed-asset and receivables variables are retained for profiling only.
- Standard KMeans on a fixed sample is used for comparable K evaluation.
- Business viability requires a minimum cluster share, a maximum cluster share,
  and sufficient normalized cluster entropy.

This notebook answers four questions:

1. Which K best balances statistical separation, resampling stability and usable cluster sizes?
2. Do the broad eligible cohort and the complete-core cohort support a similar K?
3. Can every cluster be translated into an interpretable financial archetype for Tableau?
4. Are period-t clusters associated with different period-t+1 financial outcomes?

**Leakage controls**

- Tables 01, 02 and 05 provide current-snapshot features, controls and profile variables.
- Period-t+1 fields from tables 03 and 04 are never used as clustering features.
- BB / SME / Mid Corporate labels, news and hiring signals are excluded from clustering.

## 0. Experimental design

### Clustering time point

Each company is represented by the latest account period recorded in tables 01 and 02.
Therefore, one company receives one current financial-cluster assignment in this experiment.

### Analysis cohorts

- **broad_eligible:** useful financial evidence, at least four core proxy fields,
  accounts no older than 24 months, no impossible negative values, and at least
  five available model features.
- **complete_core:** the same rules plus complete core financial fields.

`broad_eligible` is the primary business population. `complete_core` is a
sensitivity cohort used to test whether missingness changes the selected K.

### Model feature blocks

- **Scale:** current assets, creditors and employees.
- **Solvency:** equity, net assets, and total assets less current liabilities.
- **Creditor pressure:** creditors relative to current assets, plus the signed
  current-assets-minus-creditors proxy.

The three blocks receive explicit total weights of 40%, 40% and 20%.
Sparse cash, debtors and fixed-assets information is used after clustering for
profile interpretation instead of controlling company-to-company distance.

In [1]:
from __future__ import annotations

import json
import math
import os
import warnings
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path

import boto3
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid", context="notebook")

print("pandas:", pd.__version__)
import sklearn
print("scikit-learn:", sklearn.__version__)

pandas: 2.3.3
scikit-learn: 1.7.2


In [2]:
# ---------- AWS and local paths ----------
AWS_REGION = "eu-north-1"
S3_BUCKET = "team6-project-288469846191-eu-north-1-an"

S3_INPUT_PREFIX = "processed/financial_features/"
S3_OUTPUT_PREFIX = "processed/financial_clustering/"
S3_MODEL_PREFIX = "models/financial_clustering/k_selection/"
S3_NOTEBOOK_KEY = (
    "notebooks/financial_clustering/"
    "01_financial_cluster_k_selection_v2.ipynb"
)

LOCAL_ROOT = Path("/data/signalbridge")
LOCAL_INPUT_DIR = LOCAL_ROOT / "input/financial_five_tables"
LOCAL_OUTPUT_DIR = (
    LOCAL_ROOT / "output/financial_clustering/k_selection_v2"
)
LOCAL_MODEL_DIR = (
    LOCAL_ROOT / "models/financial_clustering/k_selection/v2"
)
LOCAL_NOTEBOOK_PATH = (
    LOCAL_ROOT / "notebooks/financial_clustering/"
    "01_financial_cluster_k_selection_v2.ipynb"
)

# A unique run directory prevents accidental overwriting.
RUN_ID = os.getenv(
    "SIGNALBRIDGE_RUN_ID",
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
)
LOCAL_RUN_OUTPUT_DIR = LOCAL_OUTPUT_DIR / f"run_id={RUN_ID}"
LOCAL_RUN_MODEL_DIR = LOCAL_MODEL_DIR / f"run_id={RUN_ID}"
LOCAL_FIGURE_DIR = LOCAL_RUN_OUTPUT_DIR / "figures"

S3_OUTPUT_RUN_PREFIX = (
    f"{S3_OUTPUT_PREFIX}k_selection_v2/run_id={RUN_ID}/"
)
S3_MODEL_RUN_PREFIX = (
    f"{S3_MODEL_PREFIX}v2/run_id={RUN_ID}/"
)

# ---------- K-selection settings ----------
RANDOM_STATE = 42
K_VALUES = list(range(3, 9))
PRIMARY_COHORT = "broad_eligible"

MIN_CLUSTER_SHARE = 0.02
MAX_CLUSTER_SHARE = 0.65
MIN_CLUSTER_ENTROPY = 0.70
MIN_STABILITY_ARI = 0.55

FIT_SAMPLE_SIZE = 60_000
SILHOUETTE_SAMPLE_SIZE = 8_000
STABILITY_REFERENCE_SIZE = 8_000
STABILITY_FIT_SIZE = 30_000
STABILITY_REPEATS = 3
N_INIT = 30
STABILITY_N_INIT = 10
MAX_ITER = 500

# Set an integer from 3 to 8 only after reviewing the metric dashboard.
SELECTED_K_OVERRIDE = None

UPLOAD_TO_S3 = True
FORCE_DOWNLOAD = False

MIN_GAP_DAYS = 270
MAX_GAP_DAYS = 550
MIN_CURRENT_ASSETS_FOR_RATIO = 1_000

for path in [
    LOCAL_INPUT_DIR,
    LOCAL_RUN_OUTPUT_DIR,
    LOCAL_RUN_MODEL_DIR,
    LOCAL_FIGURE_DIR,
    LOCAL_NOTEBOOK_PATH.parent,
]:
    path.mkdir(parents=True, exist_ok=True)

print("RUN_ID:", RUN_ID)
print("S3 input:", f"s3://{S3_BUCKET}/{S3_INPUT_PREFIX}")
print("S3 output:", f"s3://{S3_BUCKET}/{S3_OUTPUT_RUN_PREFIX}")
print("S3 models:", f"s3://{S3_BUCKET}/{S3_MODEL_RUN_PREFIX}")

RUN_ID: 20260725T153022Z
S3 input: s3://team6-project-288469846191-eu-north-1-an/processed/financial_features/
S3 output: s3://team6-project-288469846191-eu-north-1-an/processed/financial_clustering/k_selection_v2/run_id=20260725T153022Z/
S3 models: s3://team6-project-288469846191-eu-north-1-an/models/financial_clustering/k_selection/v2/run_id=20260725T153022Z/


## 1. Synchronise the five financial tables from S3

The notebook uses the EC2 IAM role and does not store access keys.
Existing local files are reused unless `FORCE_DOWNLOAD=True`.

In [3]:
INPUT_FILES = [
    "01_financial_status_labels_100k.csv",
    "02_financial_scale_labels_100k.csv",
    "03_financial_change_labels.csv",
    "04_financial_transition_labels.csv",
    "05_financial_data_quality_labels_100k.csv",
]

s3 = boto3.client("s3", region_name=AWS_REGION)


def download_input(filename: str) -> Path:
    local_path = LOCAL_INPUT_DIR / filename
    s3_key = f"{S3_INPUT_PREFIX}{filename}"
    if FORCE_DOWNLOAD or not local_path.exists():
        print(f"Downloading s3://{S3_BUCKET}/{s3_key}")
        s3.download_file(S3_BUCKET, s3_key, str(local_path))
    else:
        print(f"Using cached file: {local_path}")
    return local_path


input_paths = {name: download_input(name) for name in INPUT_FILES}

input_inventory = pd.DataFrame(
    [
        {
            "filename": name,
            "local_path": str(path),
            "size_mb": round(path.stat().st_size / 1024**2, 2),
        }
        for name, path in input_paths.items()
    ]
)
display(input_inventory)

Using cached file: /data/signalbridge/input/financial_five_tables/01_financial_status_labels_100k.csv
Using cached file: /data/signalbridge/input/financial_five_tables/02_financial_scale_labels_100k.csv
Using cached file: /data/signalbridge/input/financial_five_tables/03_financial_change_labels.csv
Using cached file: /data/signalbridge/input/financial_five_tables/04_financial_transition_labels.csv
Using cached file: /data/signalbridge/input/financial_five_tables/05_financial_data_quality_labels_100k.csv


,filename,local_path,size_mb
0,01_financial_status_labels_100k.csv,/data/signalbridge/input/financial_five_tables...,48.49
1,02_financial_scale_labels_100k.csv,/data/signalbridge/input/financial_five_tables...,57.94
2,03_financial_change_labels.csv,/data/signalbridge/input/financial_five_tables...,76.95
3,04_financial_transition_labels.csv,/data/signalbridge/input/financial_five_tables...,21.68
4,05_financial_data_quality_labels_100k.csv,/data/signalbridge/input/financial_five_tables...,52.20


## 2. Load the current snapshot and audit the joins

`CompanyNumber_norm` is always read as a string to preserve leading zeros.
Each company-level table must contain one row per company, enforced through
`validate="one_to_one"` during the joins.

In [4]:
ID = "CompanyNumber_norm"

STATUS_COLUMNS = [
    ID,
    "CompanyName",
    "primary_sector",
    "Accounts_AccountCategory",
    "latest_period_end",
    "latest_available_date",
    "financial_evidence_tier",
    "cash",
    "creditors_total",
    "current_assets",
    "debtors",
    "employees",
    "equity",
    "fixed_assets",
    "net_assets_liabilities",
    "net_current_assets_liabilities",
    "profit_loss",
    "total_assets_less_current_liabilities",
    "negative_equity_flag",
    "working_capital_deficit_flag",
    "reported_loss_flag",
    "creditors_exceed_current_assets_flag",
]

SCALE_COLUMNS = [
    ID,
    "current_assets_scale_band",
    "fixed_assets_scale_band",
    "creditors_scale_band",
    "absolute_equity_scale_band",
    "employee_scale_band",
    "net_assets_scale_band",
    "total_assets_scale_band",
    "financial_scale_available_field_count",
]

QUALITY_COLUMNS = [
    ID,
    "available_core_proxy_field_count",
    "available_any_financial_field_count",
    "matched_account_periods",
    "useful_financial_periods",
    "has_two_plus_financial_periods_flag",
    "useful_financial_evidence_flag",
    "core_fields_complete_flag",
    "accounts_older_than_24m_flag",
    "impossible_negative_value_flag",
    "extreme_amount_p999_flag",
    "latest_period_gap_anomaly_flag",
    "evidence_improved_flag",
    "evidence_deteriorated_flag",
]

read_kwargs = {"dtype": {ID: "string"}, "low_memory": False}
status = pd.read_csv(
    input_paths["01_financial_status_labels_100k.csv"],
    usecols=STATUS_COLUMNS,
    **read_kwargs,
)
scale = pd.read_csv(
    input_paths["02_financial_scale_labels_100k.csv"],
    usecols=SCALE_COLUMNS,
    **read_kwargs,
)
quality = pd.read_csv(
    input_paths["05_financial_data_quality_labels_100k.csv"],
    usecols=QUALITY_COLUMNS,
    **read_kwargs,
)

for frame_name, frame in {
    "status": status,
    "scale": scale,
    "quality": quality,
}.items():
    duplicate_count = int(frame[ID].duplicated().sum())
    assert duplicate_count == 0, f"{frame_name} contains duplicate company IDs"
    print(frame_name, frame.shape, "unique companies:", frame[ID].nunique())

current = (
    status.merge(scale, on=ID, how="left", validate="one_to_one")
    .merge(quality, on=ID, how="left", validate="one_to_one")
)
current["latest_period_end"] = pd.to_datetime(
    current["latest_period_end"], errors="coerce"
)
current["latest_available_date"] = pd.to_datetime(
    current["latest_available_date"], errors="coerce"
)

assert len(current) == len(status)
print("Merged current snapshot:", current.shape)
display(
    current[
        [
            ID,
            "latest_period_end",
            "financial_evidence_tier",
            "available_core_proxy_field_count",
            "core_fields_complete_flag",
        ]
    ].head()
)

status (100000, 22) unique companies: 100000
scale (100000, 9) unique companies: 100000
quality (100000, 14) unique companies: 100000
Merged current snapshot: (100000, 43)


,CompanyNumber_norm,latest_period_end,financial_evidence_tier,available_core_proxy_field_count,core_fields_complete_flag
0,13209628,2025-06-30,T2_balance_sheet_rich,7.0,True
1,13887674,2025-02-28,T2_balance_sheet_rich,4.0,False
2,13408899,NaT,NaN,NaN,False
3,SC374368,2025-03-31,T2_balance_sheet_rich,7.0,True
4,13675979,2025-10-31,T2_balance_sheet_rich,5.0,False


In [5]:
audit = pd.DataFrame(
    {
        "metric": [
            "all_companies",
            "useful_financial_evidence",
            "core_fields_complete",
            "two_plus_financial_periods",
            "accounts_older_than_24m",
            "impossible_negative_value",
            "extreme_amount_p999",
        ],
        "count": [
            len(current),
            current["useful_financial_evidence_flag"].fillna(False).sum(),
            current["core_fields_complete_flag"].fillna(False).sum(),
            current["has_two_plus_financial_periods_flag"].fillna(False).sum(),
            current["accounts_older_than_24m_flag"].fillna(False).sum(),
            current["impossible_negative_value_flag"].fillna(False).sum(),
            current["extreme_amount_p999_flag"].fillna(False).sum(),
        ],
    }
)
audit["share"] = audit["count"] / len(current)
display(audit)

display(
    current["financial_evidence_tier"]
    .value_counts(dropna=False)
    .rename_axis("financial_evidence_tier")
    .to_frame("companies")
)

,metric,count,share
0,all_companies,100000,1.00000
1,useful_financial_evidence,93760,0.93760
2,core_fields_complete,23479,0.23479
3,two_plus_financial_periods,76423,0.76423
4,accounts_older_than_24m,1125,0.01125
5,impossible_negative_value,753,0.00753
6,extreme_amount_p999,229,0.00229


,companies
financial_evidence_tier,
T2_balance_sheet_rich,88095
NaN,5688
T3_balance_sheet_partial,3300
T1_observed_turnover,2365
T4_account_category_only,552


## 3. Engineer features that can be reconstructed at historical period t

The model uses only variables that can also be reconstructed from table 03 at
historical period t. Amounts receive log or signed-log transformations.

The two ratios that dominated V1 are removed:

- employees per million disclosed assets;
- creditors to disclosed assets.

The remaining creditor-pressure ratio is log-transformed and is eligible only
when current assets are at least GBP 1,000. This prevents tiny denominators from
creating extreme distances.

Data-quality fields, sector, Account Category and existing Low/Medium/High labels
do not enter the distance calculation. They are used for filtering, interpretation
and sensitivity analysis.

In [6]:
RAW_BASE_FIELDS = [
    "cash",
    "creditors_total",
    "current_assets",
    "debtors",
    "employees",
    "equity",
    "fixed_assets",
    "net_assets_liabilities",
    "total_assets_less_current_liabilities",
]

FEATURE_GROUPS = {
    "scale": [
        "log_current_assets",
        "log_creditors_total",
        "log_employees",
    ],
    "solvency": [
        "signed_log_equity",
        "signed_log_net_assets_liabilities",
        "signed_log_total_assets_less_current_liabilities",
    ],
    "creditor_pressure": [
        "log_creditors_to_current_assets_proxy",
        "signed_log_current_assets_minus_creditors_proxy",
    ],
}
FEATURE_GROUP_IMPORTANCE = {
    "scale": 0.40,
    "solvency": 0.40,
    "creditor_pressure": 0.20,
}
MODEL_FEATURES = [
    feature
    for group_features in FEATURE_GROUPS.values()
    for feature in group_features
]


def safe_log1p(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    return np.log1p(values.clip(lower=0))


def signed_log1p(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    return np.sign(values) * np.log1p(np.abs(values))


def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=frame.index)

    out["log_current_assets"] = safe_log1p(frame["current_assets"])
    out["log_creditors_total"] = safe_log1p(frame["creditors_total"])
    out["log_employees"] = safe_log1p(frame["employees"])

    for field in [
        "equity",
        "net_assets_liabilities",
        "total_assets_less_current_liabilities",
    ]:
        out[f"signed_log_{field}"] = signed_log1p(frame[field])

    current_assets = pd.to_numeric(
        frame["current_assets"], errors="coerce"
    )
    creditors = pd.to_numeric(
        frame["creditors_total"], errors="coerce"
    )
    ratio_eligible = (
        current_assets.ge(MIN_CURRENT_ASSETS_FOR_RATIO)
        & creditors.ge(0)
    )
    creditor_ratio = (creditors / current_assets).where(ratio_eligible)
    out["log_creditors_to_current_assets_proxy"] = np.log1p(
        creditor_ratio.clip(lower=0)
    )
    out["signed_log_current_assets_minus_creditors_proxy"] = (
        signed_log1p((current_assets - creditors).where(ratio_eligible))
    )

    return out[MODEL_FEATURES].replace([np.inf, -np.inf], np.nan)


engineered = engineer_features(current)
current["model_feature_nonmissing_count"] = (
    engineered.notna().sum(axis=1)
)

feature_coverage = (
    engineered.notna()
    .mean()
    .rename("coverage")
    .sort_values()
    .to_frame()
)
display(feature_coverage)

,coverage
log_creditors_to_current_assets_proxy,0.71324
signed_log_current_assets_minus_creditors_proxy,0.71324
signed_log_net_assets_liabilities,0.81057
log_current_assets,0.83293
signed_log_total_assets_less_current_liabilities,0.84702
log_creditors_total,0.88741
log_employees,0.90359
signed_log_equity,0.91479


In [7]:
base_quality_mask = (
    current["useful_financial_evidence_flag"].fillna(False)
    & current["available_core_proxy_field_count"].fillna(0).ge(4)
    & ~current["accounts_older_than_24m_flag"].fillna(True)
    & ~current["impossible_negative_value_flag"].fillna(True)
    & current["model_feature_nonmissing_count"].ge(5)
)

cohort_masks = {
    "broad_eligible": base_quality_mask,
    "complete_core": (
        base_quality_mask
        & current["core_fields_complete_flag"].fillna(False)
    ),
}

cohort_summary = pd.DataFrame(
    [
        {
            "cohort": name,
            "companies": int(mask.sum()),
            "share_of_all": float(mask.mean()),
            "median_available_model_features": float(
                current.loc[
                    mask, "model_feature_nonmissing_count"
                ].median()
            ),
        }
        for name, mask in cohort_masks.items()
    ]
)
display(cohort_summary)

assert cohort_masks[PRIMARY_COHORT].sum() >= 10_000, (
    "Primary cohort unexpectedly small; inspect quality filters."
)

,cohort,companies,share_of_all,median_available_model_features
0,broad_eligible,84711,0.84711,8.0
1,complete_core,22878,0.22878,8.0


## 4. Preprocessing: winsorisation, imputation, standardisation and block weights

Preprocessing is fitted only on the primary cohort and then held fixed for
the complete-core cohort and historical period-t observations.

V2 uses StandardScaler after log transformation and 1st/99th percentile
clipping. The feature-block weights sum to one at the block level:
40% scale, 40% solvency and 20% creditor pressure.

In [8]:
primary_index = current.index[cohort_masks[PRIMARY_COHORT]]
primary_features = engineered.loc[
    primary_index, MODEL_FEATURES
].copy()

clip_bounds = pd.DataFrame(
    {
        "lower": primary_features.quantile(0.01),
        "upper": primary_features.quantile(0.99),
    }
)


def clip_features(frame: pd.DataFrame) -> pd.DataFrame:
    clipped = frame[MODEL_FEATURES].copy()
    for feature in MODEL_FEATURES:
        clipped[feature] = clipped[feature].clip(
            lower=clip_bounds.loc[feature, "lower"],
            upper=clip_bounds.loc[feature, "upper"],
        )
    return clipped


imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

primary_clipped = clip_features(primary_features)
primary_imputed = imputer.fit_transform(primary_clipped)
scaler.fit(primary_imputed)

feature_weights = pd.Series(index=MODEL_FEATURES, dtype=float)
for group, group_features in FEATURE_GROUPS.items():
    block_importance = FEATURE_GROUP_IMPORTANCE[group]
    feature_weights.loc[group_features] = math.sqrt(
        block_importance / len(group_features)
    )


def transform_features(frame: pd.DataFrame) -> np.ndarray:
    clipped = clip_features(frame)
    imputed = imputer.transform(clipped)
    scaled = scaler.transform(imputed)
    return scaled * feature_weights.loc[MODEL_FEATURES].to_numpy()


X_by_cohort = {
    name: transform_features(
        engineered.loc[mask, MODEL_FEATURES]
    )
    for name, mask in cohort_masks.items()
}

preprocessing_artifact = {
    "model_features": MODEL_FEATURES,
    "feature_groups": FEATURE_GROUPS,
    "feature_group_importance": FEATURE_GROUP_IMPORTANCE,
    "feature_weights": feature_weights.to_dict(),
    "clip_bounds": clip_bounds.to_dict(orient="index"),
    "imputer": imputer,
    "scaler": scaler,
    "quality_rule": {
        "useful_financial_evidence_flag": True,
        "minimum_available_core_proxy_fields": 4,
        "accounts_older_than_24m_flag": False,
        "impossible_negative_value_flag": False,
        "minimum_nonmissing_model_features": 5,
        "minimum_current_assets_for_ratio": (
            MIN_CURRENT_ASSETS_FOR_RATIO
        ),
    },
}

feature_preprocessing_summary = pd.DataFrame(
    {
        "feature": MODEL_FEATURES,
        "group": [
            group
            for group, features in FEATURE_GROUPS.items()
            for _ in features
        ],
        "distance_weight": (
            feature_weights.loc[MODEL_FEATURES].values
        ),
        "clip_lower": (
            clip_bounds.loc[MODEL_FEATURES, "lower"].values
        ),
        "clip_upper": (
            clip_bounds.loc[MODEL_FEATURES, "upper"].values
        ),
    }
)
display(feature_preprocessing_summary)

,feature,group,distance_weight,clip_lower,clip_upper
0,log_current_assets,scale,0.365148,0.693147,15.529080
1,log_creditors_total,scale,0.365148,0.000000,14.958937
2,log_employees,scale,0.365148,0.000000,3.784190
3,signed_log_equity,solvency,0.365148,-12.683113,15.338761
4,signed_log_net_assets_liabilities,solvency,0.365148,-12.752029,15.412607
5,signed_log_total_assets_less_current_liabilities,solvency,0.365148,-12.181367,15.749423
6,log_creditors_to_current_assets_proxy,creditor_pressure,0.316228,0.000000,4.630924
7,signed_log_current_assets_minus_creditors_proxy,creditor_pressure,0.316228,-14.162969,15.164735


## 5. Select K using separation, stability and business viability

Metrics:

- **Silhouette:** higher is better.
- **Calinski-Harabasz:** higher is better.
- **Davies-Bouldin:** lower is better.
- **Bootstrap ARI:** measures whether independently resampled models assign
  the same reference companies consistently.
- **Minimum and maximum cluster shares:** prevent tiny outlier clusters and
  one dominant catch-all cluster.
- **Normalized entropy:** measures overall cluster-size balance.
- **Inertia:** should generally decrease as K increases when models are fitted
  comparably.

Standard KMeans is fitted to one fixed sample per cohort for every K.
This makes inertia and quality metrics more comparable than V1's separate
MiniBatchKMeans solutions.

In [9]:
def make_clusterer(
    k: int,
    seed: int,
    n_init: int = N_INIT,
) -> KMeans:
    return KMeans(
        n_clusters=k,
        random_state=seed,
        n_init=n_init,
        max_iter=MAX_ITER,
        algorithm="lloyd",
    )


def sample_indices(
    n_rows: int,
    size: int,
    rng: np.random.Generator,
    replace: bool = False,
) -> np.ndarray:
    actual_size = min(size, n_rows) if not replace else size
    return rng.choice(
        n_rows, size=actual_size, replace=replace
    )


def normalized_cluster_entropy(
    labels: np.ndarray,
    k: int,
) -> float:
    shares = np.bincount(labels, minlength=k) / len(labels)
    positive = shares[shares > 0]
    entropy = -(positive * np.log(positive)).sum()
    return float(entropy / np.log(k))


def evaluate_k_for_cohort(
    X: np.ndarray,
    cohort_name: str,
) -> pd.DataFrame:
    rng = np.random.default_rng(RANDOM_STATE)
    fit_idx = sample_indices(
        len(X), FIT_SAMPLE_SIZE, rng, replace=False
    )
    eval_idx = sample_indices(
        len(X), SILHOUETTE_SAMPLE_SIZE, rng, replace=False
    )
    stability_ref_idx = sample_indices(
        len(X), STABILITY_REFERENCE_SIZE, rng, replace=False
    )

    X_fit = X[fit_idx]
    X_eval = X[eval_idx]
    X_stability_ref = X[stability_ref_idx]
    rows = []

    for k in K_VALUES:
        print(f"[{cohort_name}] evaluating K={k}")
        model = make_clusterer(k, RANDOM_STATE)
        model.fit(X_fit)

        labels_all = model.predict(X)
        labels_eval = model.predict(X_eval)
        shares = (
            np.bincount(labels_all, minlength=k) / len(labels_all)
        )

        repeat_predictions = []
        for repeat in range(STABILITY_REPEATS):
            repeat_rng = np.random.default_rng(
                RANDOM_STATE + 10_000 * k + repeat
            )
            boot_idx = sample_indices(
                len(X_fit),
                min(STABILITY_FIT_SIZE, len(X_fit)),
                repeat_rng,
                replace=True,
            )
            repeat_model = make_clusterer(
                k,
                RANDOM_STATE + 1_000 + repeat,
                n_init=STABILITY_N_INIT,
            )
            repeat_model.fit(X_fit[boot_idx])
            repeat_predictions.append(
                repeat_model.predict(X_stability_ref)
            )

        pairwise_ari = [
            adjusted_rand_score(a, b)
            for a, b in combinations(repeat_predictions, 2)
        ]

        rows.append(
            {
                "cohort": cohort_name,
                "k": k,
                "n_companies": len(X),
                "fit_sample_size": len(X_fit),
                "inertia_per_fit_company": (
                    model.inertia_ / len(X_fit)
                ),
                "silhouette": silhouette_score(
                    X_eval, labels_eval
                ),
                "calinski_harabasz": (
                    calinski_harabasz_score(X_eval, labels_eval)
                ),
                "davies_bouldin": davies_bouldin_score(
                    X_eval, labels_eval
                ),
                "bootstrap_ari_mean": float(
                    np.mean(pairwise_ari)
                ),
                "bootstrap_ari_min": float(
                    np.min(pairwise_ari)
                ),
                "min_cluster_share": float(shares.min()),
                "max_cluster_share": float(shares.max()),
                "normalized_cluster_entropy": (
                    normalized_cluster_entropy(labels_all, k)
                ),
            }
        )

    return pd.DataFrame(rows)


k_metrics = pd.concat(
    [
        evaluate_k_for_cohort(X, cohort_name)
        for cohort_name, X in X_by_cohort.items()
    ],
    ignore_index=True,
)
display(k_metrics)

[broad_eligible] evaluating K=3
[broad_eligible] evaluating K=4
[broad_eligible] evaluating K=5
[broad_eligible] evaluating K=6
[broad_eligible] evaluating K=7
[broad_eligible] evaluating K=8
[complete_core] evaluating K=3
[complete_core] evaluating K=4
[complete_core] evaluating K=5
[complete_core] evaluating K=6
[complete_core] evaluating K=7
[complete_core] evaluating K=8


,cohort,k,n_companies,fit_sample_size,inertia_per_fit_company,silhouette,calinski_harabasz,davies_bouldin,bootstrap_ari_mean,bootstrap_ari_min,min_cluster_share,max_cluster_share,normalized_cluster_entropy
0,broad_eligible,3,84711,60000,0.578373,0.220417,2875.786236,1.407387,0.973006,0.959614,0.194213,0.484317,0.941401
1,broad_eligible,4,84711,60000,0.487203,0.258846,2758.496387,1.205293,0.964986,0.949359,0.148989,0.381108,0.952874
2,broad_eligible,5,84711,60000,0.423393,0.289062,2718.058807,1.151507,0.981504,0.975125,0.112665,0.410029,0.924112
3,broad_eligible,6,84711,60000,0.377428,0.302129,2637.009094,1.226478,0.977596,0.968869,0.067358,0.405355,0.900164
4,broad_eligible,7,84711,60000,0.345109,0.263781,2522.472368,1.247759,0.981638,0.979092,0.054385,0.319888,0.916416
5,broad_eligible,8,84711,60000,0.314590,0.260099,2490.979487,1.200450,0.971030,0.968596,0.050064,0.285500,0.906114
6,complete_core,3,22878,22878,0.549351,0.302719,3547.125256,1.156113,0.981691,0.973374,0.179474,0.573040,0.885613
7,complete_core,4,22878,22878,0.458132,0.321073,3409.649837,1.135455,0.991763,0.988202,0.147565,0.449690,0.926378
8,complete_core,5,22878,22878,0.403099,0.336164,3175.562110,1.257366,0.748654,0.626474,0.087202,0.446193,0.879236
9,complete_core,6,22878,22878,0.351260,0.284698,3145.575343,1.221493,0.943284,0.918886,0.082131,0.315500,0.938985


In [10]:
def add_composite_score(group: pd.DataFrame) -> pd.DataFrame:
    out = group.copy()

    out["score_silhouette"] = out["silhouette"].rank(pct=True)
    out["score_calinski"] = (
        out["calinski_harabasz"].rank(pct=True)
    )
    out["score_davies"] = (
        -out["davies_bouldin"]
    ).rank(pct=True)
    out["score_stability"] = (
        out["bootstrap_ari_mean"].rank(pct=True)
    )
    out["score_balance"] = (
        out["normalized_cluster_entropy"].rank(pct=True)
    )

    out["composite_score"] = (
        0.25 * out["score_silhouette"]
        + 0.10 * out["score_calinski"]
        + 0.10 * out["score_davies"]
        + 0.30 * out["score_stability"]
        + 0.25 * out["score_balance"]
    )

    out["business_size_eligible"] = (
        out["min_cluster_share"].ge(MIN_CLUSTER_SHARE)
        & out["max_cluster_share"].le(MAX_CLUSTER_SHARE)
        & out["normalized_cluster_entropy"].ge(
            MIN_CLUSTER_ENTROPY
        )
    )
    out["stability_eligible"] = out[
        "bootstrap_ari_mean"
    ].ge(MIN_STABILITY_ARI)
    out["fully_eligible"] = (
        out["business_size_eligible"]
        & out["stability_eligible"]
    )
    return out


scored_parts = []
for cohort_name, cohort_metrics in k_metrics.groupby(
    "cohort", sort=False
):
    scored = add_composite_score(cohort_metrics)
    scored_parts.append(scored)
k_metrics_scored = pd.concat(
    scored_parts, ignore_index=True
)

shortlist = (
    k_metrics_scored.sort_values(
        ["cohort", "fully_eligible", "composite_score"],
        ascending=[True, False, False],
    )
    .groupby("cohort", as_index=False)
    .head(3)
)

primary_metrics = k_metrics_scored[
    k_metrics_scored["cohort"] == PRIMARY_COHORT
].copy()
primary_eligible = primary_metrics[
    primary_metrics["fully_eligible"]
].sort_values("composite_score", ascending=False)

if primary_eligible.empty:
    AUTOMATIC_RECOMMENDED_K = int(
        primary_metrics.sort_values(
            "composite_score", ascending=False
        ).iloc[0]["k"]
    )
    RECOMMENDATION_STATUS = (
        "No K passed all business and stability rules. "
        "The displayed K is diagnostic only and requires review."
    )
else:
    AUTOMATIC_RECOMMENDED_K = int(
        primary_eligible.iloc[0]["k"]
    )
    RECOMMENDATION_STATUS = (
        "The recommended K passed all configured rules."
    )

SELECTED_K = (
    int(SELECTED_K_OVERRIDE)
    if SELECTED_K_OVERRIDE is not None
    else AUTOMATIC_RECOMMENDED_K
)

if SELECTED_K not in K_VALUES:
    raise ValueError(f"SELECTED_K must be one of {K_VALUES}")

selected_row = primary_metrics[
    primary_metrics["k"] == SELECTED_K
].iloc[0]
SELECTED_K_PASSES_ALL_RULES = bool(
    selected_row["fully_eligible"]
)

print("Automatic recommended K:", AUTOMATIC_RECOMMENDED_K)
print("Recommendation status:", RECOMMENDATION_STATUS)
print("K used for candidate model:", SELECTED_K)
print(
    "Selected K passes all rules:",
    SELECTED_K_PASSES_ALL_RULES,
)
display(shortlist)

Automatic recommended K: 5
Recommendation status: The recommended K passed all configured rules.
K used for candidate model: 5
Selected K passes all rules: True


,cohort,k,n_companies,fit_sample_size,inertia_per_fit_company,silhouette,calinski_harabasz,davies_bouldin,bootstrap_ari_mean,bootstrap_ari_min,min_cluster_share,max_cluster_share,normalized_cluster_entropy,score_silhouette,score_calinski,score_davies,score_stability,score_balance,composite_score,business_size_eligible,stability_eligible,fully_eligible
2,broad_eligible,5,84711,60000,0.423393,0.289062,2718.058807,1.151507,0.981504,0.975125,0.112665,0.410029,0.924112,0.833333,0.666667,1.000000,0.833333,0.666667,0.791667,True,True,True
4,broad_eligible,7,84711,60000,0.345109,0.263781,2522.472368,1.247759,0.981638,0.979092,0.054385,0.319888,0.916416,0.666667,0.333333,0.333333,1.000000,0.500000,0.658333,True,True,True
3,broad_eligible,6,84711,60000,0.377428,0.302129,2637.009094,1.226478,0.977596,0.968869,0.067358,0.405355,0.900164,1.000000,0.500000,0.500000,0.666667,0.166667,0.591667,True,True,True
7,complete_core,4,22878,22878,0.458132,0.321073,3409.649837,1.135455,0.991763,0.988202,0.147565,0.449690,0.926378,0.833333,0.833333,1.000000,1.000000,0.666667,0.858333,True,True,True
6,complete_core,3,22878,22878,0.549351,0.302719,3547.125256,1.156113,0.981691,0.973374,0.179474,0.573040,0.885613,0.666667,1.000000,0.833333,0.833333,0.333333,0.683333,True,True,True
9,complete_core,6,22878,22878,0.351260,0.284698,3145.575343,1.221493,0.943284,0.918886,0.082131,0.315500,0.938985,0.500000,0.500000,0.333333,0.500000,1.000000,0.608333,True,True,True


In [ ]:
metric_plot_specs = [
    ("inertia_per_fit_company", "Inertia per company", False),
    ("silhouette", "Silhouette", True),
    ("calinski_harabasz", "Calinski–Harabasz", True),
    ("davies_bouldin", "Davies–Bouldin", False),
    ("bootstrap_ari_mean", "Bootstrap stability (ARI)", True),
    ("min_cluster_share", "Minimum cluster share", True),
]

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
for ax, (metric, title, higher_is_better) in zip(
    axes.flat, metric_plot_specs
):
    sns.lineplot(
        data=k_metrics_scored,
        x="k",
        y=metric,
        hue="cohort",
        marker="o",
        ax=ax,
    )
    ax.axvline(
        SELECTED_K,
        color="black",
        linestyle="--",
        alpha=0.6,
        label=f"selected K={SELECTED_K}",
    )
    if metric == "min_cluster_share":
        ax.axhline(
            MIN_CLUSTER_SHARE,
            color="red",
            linestyle=":",
            alpha=0.7,
            label="minimum acceptable share",
        )
    ax.set_title(title)
    ax.set_xticks(K_VALUES)

plt.tight_layout()
k_selection_figure = LOCAL_FIGURE_DIR / "k_selection_dashboard.png"
fig.savefig(k_selection_figure, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(
    data=k_metrics_scored,
    x="k",
    y="composite_score",
    hue="cohort",
    marker="o",
    ax=ax,
)
ax.axvline(SELECTED_K, color="black", linestyle="--")
ax.set_title("Composite K-selection score")
ax.set_xticks(K_VALUES)
plt.tight_layout()
composite_figure = LOCAL_FIGURE_DIR / "k_composite_score.png"
fig.savefig(composite_figure, dpi=180, bbox_inches="tight")
plt.show()

### Manual review rules

The automatic recommendation is a shortlist, not a substitute for business review.
The final K should satisfy all of the following:

1. The broad and complete-core cohorts do not strongly contradict each other.
2. Bootstrap ARI does not collapse at the selected K.
3. Every cluster covers at least 2% of the primary cohort.
4. No cluster covers more than 65% of the primary cohort.
5. Normalized cluster entropy is at least 0.70.
6. Every cluster can be explained with two or three financial characteristics.
7. At least some clusters show meaningful differences in subsequent outcomes.

To apply a reviewed choice, set `SELECTED_K_OVERRIDE` near the top and rerun
from the K-selection section.

## 6. Fit the candidate model and produce company-level assignments

The final candidate uses standard KMeans on one fixed sample from the primary
cohort. All eligible companies are then assigned to the nearest centroid.

Cluster IDs are reordered by the combined scale-centroid score and reported as
`FC01...FCK`, reducing arbitrary label changes between runs.

In [ ]:
X_primary = X_by_cohort[PRIMARY_COHORT]
final_rng = np.random.default_rng(RANDOM_STATE)
final_fit_idx = sample_indices(
    len(X_primary),
    FIT_SAMPLE_SIZE,
    final_rng,
    replace=False,
)
final_model = make_clusterer(SELECTED_K, RANDOM_STATE)
final_model.fit(X_primary[final_fit_idx])

original_labels = final_model.predict(X_primary)
distances = final_model.transform(X_primary)

own_distance = distances[
    np.arange(len(distances)), original_labels
]
sorted_distances = np.sort(distances, axis=1)
assignment_margin = (
    sorted_distances[:, 1] - sorted_distances[:, 0]
) / (sorted_distances[:, 1] + 1e-12)

centers_unweighted = (
    final_model.cluster_centers_
    / feature_weights.loc[MODEL_FEATURES].to_numpy()
)
scale_indices = [
    MODEL_FEATURES.index(feature)
    for feature in FEATURE_GROUPS["scale"]
]
cluster_size_index = centers_unweighted[
    :, scale_indices
].mean(axis=1)
ordered_old_labels = np.argsort(cluster_size_index)
old_to_new = {
    int(old): int(new)
    for new, old in enumerate(ordered_old_labels, start=1)
}
new_numeric_labels = np.array(
    [old_to_new[int(label)] for label in original_labels]
)

assignments = current.loc[
    primary_index,
    [
        ID,
        "CompanyName",
        "primary_sector",
        "Accounts_AccountCategory",
        "latest_period_end",
        "latest_available_date",
        "financial_evidence_tier",
        "available_core_proxy_field_count",
        "model_feature_nonmissing_count",
        "core_fields_complete_flag",
        "extreme_amount_p999_flag",
    ],
].copy()
assignments["cluster_number"] = new_numeric_labels
assignments["financial_cluster_id"] = assignments[
    "cluster_number"
].map(lambda value: f"FC{value:02d}")
assignments["cluster_distance"] = own_distance
assignments["assignment_margin"] = assignment_margin
assignments["imputed_feature_count"] = (
    len(MODEL_FEATURES)
    - engineered.loc[
        primary_index, MODEL_FEATURES
    ].notna().sum(axis=1)
).to_numpy()
assignments["cluster_assignment_confidence"] = pd.cut(
    assignments["assignment_margin"],
    bins=[-np.inf, 0.10, 0.25, np.inf],
    labels=["Low", "Medium", "High"],
)

cluster_assignment_summary = (
    assignments["financial_cluster_id"]
    .value_counts()
    .sort_index()
    .rename("companies")
    .to_frame()
)
cluster_assignment_summary["company_share"] = (
    cluster_assignment_summary["companies"] / len(assignments)
)
display(cluster_assignment_summary)
display(assignments.head())

In [ ]:
PROFILE_VALUE_FIELDS = [
    "cash",
    "creditors_total",
    "current_assets",
    "debtors",
    "employees",
    "equity",
    "fixed_assets",
    "net_assets_liabilities",
    "net_current_assets_liabilities",
    "total_assets_less_current_liabilities",
]
PROFILE_FLAG_FIELDS = [
    "negative_equity_flag",
    "working_capital_deficit_flag",
    "reported_loss_flag",
    "creditors_exceed_current_assets_flag",
]

profile_source = assignments[[ID, "financial_cluster_id"]].merge(
    current[
        [ID]
        + PROFILE_VALUE_FIELDS
        + PROFILE_FLAG_FIELDS
        + [
            "core_fields_complete_flag",
            "has_two_plus_financial_periods_flag",
        ]
    ],
    on=ID,
    how="left",
    validate="one_to_one",
)

cluster_counts = (
    assignments.groupby("financial_cluster_id")
    .size()
    .rename("companies")
)
profile_medians = profile_source.groupby(
    "financial_cluster_id"
)[PROFILE_VALUE_FIELDS].median()
profile_flag_rates = profile_source.groupby(
    "financial_cluster_id"
)[PROFILE_FLAG_FIELDS + [
    "core_fields_complete_flag",
    "has_two_plus_financial_periods_flag",
]].mean()

cluster_profiles = (
    cluster_counts.to_frame()
    .join(profile_medians)
    .join(profile_flag_rates.add_suffix("_rate"))
    .reset_index()
)
cluster_profiles["company_share"] = (
    cluster_profiles["companies"] / len(assignments)
)

# Generate candidate descriptors from cluster medians relative to the overall median.
descriptor_fields = [
    "current_assets",
    "fixed_assets",
    "creditors_total",
    "equity",
    "net_assets_liabilities",
    "cash",
    "employees",
]
descriptor_names = {
    "current_assets": "current-assets scale",
    "fixed_assets": "fixed-assets scale",
    "creditors_total": "creditor scale",
    "equity": "equity",
    "net_assets_liabilities": "net assets",
    "cash": "cash",
    "employees": "employee scale",
}
global_median = profile_source[descriptor_fields].median()
global_iqr = (
    profile_source[descriptor_fields].quantile(0.75)
    - profile_source[descriptor_fields].quantile(0.25)
).replace(0, np.nan)
descriptor_z = (
    profile_medians[descriptor_fields] - global_median
) / global_iqr


def describe_cluster(cluster_id: str) -> str:
    row = descriptor_z.loc[cluster_id].dropna()
    if row.empty:
        return "Limited-information profile"
    strongest = row.abs().sort_values(ascending=False).head(2).index
    parts = [
        ("High " if row[field] >= 0 else "Low ")
        + descriptor_names[field]
        for field in strongest
    ]

    flag_row = profile_flag_rates.loc[cluster_id]
    risk_parts = []
    if flag_row.get("negative_equity_flag", 0) >= 0.35:
        risk_parts.append("elevated negative-equity rate")
    if flag_row.get("working_capital_deficit_flag", 0) >= 0.45:
        risk_parts.append("working-capital pressure")
    if flag_row.get("creditors_exceed_current_assets_flag", 0) >= 0.45:
        risk_parts.append("creditor pressure")

    return " + ".join(parts + risk_parts[:1])


cluster_profiles["cluster_name_auto"] = cluster_profiles[
    "financial_cluster_id"
].map(describe_cluster)
name_map = cluster_profiles.set_index(
    "financial_cluster_id"
)["cluster_name_auto"].to_dict()
assignments["cluster_name_auto"] = assignments[
    "financial_cluster_id"
].map(name_map)

display(
    cluster_profiles[
        [
            "financial_cluster_id",
            "cluster_name_auto",
            "companies",
            "company_share",
            "current_assets",
            "fixed_assets",
            "creditors_total",
            "equity",
            "negative_equity_flag_rate",
            "working_capital_deficit_flag_rate",
            "creditors_exceed_current_assets_flag_rate",
        ]
    ]
)

In [ ]:
profile_heatmap_data = descriptor_z.copy()
profile_heatmap_data.index = [
    f"{cluster_id}: {name_map[cluster_id]}"
    for cluster_id in profile_heatmap_data.index
]

fig, ax = plt.subplots(
    figsize=(12, max(5, 0.7 * len(profile_heatmap_data)))
)
sns.heatmap(
    profile_heatmap_data,
    cmap="RdBu_r",
    center=0,
    annot=True,
    fmt=".1f",
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Cluster profiles: robust deviation from overall median")
ax.set_xlabel("Financial profile variable")
ax.set_ylabel("Financial cluster")
plt.tight_layout()
profile_figure = LOCAL_FIGURE_DIR / "cluster_profile_heatmap.png"
fig.savefig(profile_figure, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
plot_counts = (
    assignments["financial_cluster_id"]
    .value_counts()
    .sort_index()
    .rename_axis("financial_cluster_id")
    .reset_index(name="companies")
)
sns.barplot(
    data=plot_counts,
    x="financial_cluster_id",
    y="companies",
    color="#4C78A8",
    ax=ax,
)
ax.set_title("Company count by financial cluster")
plt.tight_layout()
size_figure = LOCAL_FIGURE_DIR / "cluster_size_distribution.png"
fig.savefig(size_figure, dpi=180, bbox_inches="tight")
plt.show()

## 7. Cross-tabulations for Tableau and business analysis

Sector and Account Category do not enter the clustering model. These tables
test whether a cluster is merely a disguised industry or disclosure-category
split and provide Tableau-ready profile dimensions.

In [ ]:
cluster_sector_mix = (
    assignments.groupby(
        ["financial_cluster_id", "primary_sector"],
        dropna=False,
    )
    .size()
    .rename("companies")
    .reset_index()
)
cluster_sector_mix["share_within_cluster"] = (
    cluster_sector_mix["companies"]
    / cluster_sector_mix.groupby("financial_cluster_id")[
        "companies"
    ].transform("sum")
)

cluster_account_category_mix = (
    assignments.groupby(
        ["financial_cluster_id", "Accounts_AccountCategory"],
        dropna=False,
    )
    .size()
    .rename("companies")
    .reset_index()
)
cluster_account_category_mix["share_within_cluster"] = (
    cluster_account_category_mix["companies"]
    / cluster_account_category_mix.groupby("financial_cluster_id")[
        "companies"
    ].transform("sum")
)

scale_band_columns = [
    "current_assets_scale_band",
    "fixed_assets_scale_band",
    "creditors_scale_band",
    "absolute_equity_scale_band",
    "employee_scale_band",
    "net_assets_scale_band",
    "total_assets_scale_band",
]
scale_long = (
    assignments[[ID, "financial_cluster_id"]]
    .merge(
        current[[ID] + scale_band_columns],
        on=ID,
        how="left",
        validate="one_to_one",
    )
    .melt(
        id_vars=[ID, "financial_cluster_id"],
        value_vars=scale_band_columns,
        var_name="scale_dimension",
        value_name="scale_band",
    )
)
cluster_scale_band_mix = (
    scale_long.groupby(
        ["financial_cluster_id", "scale_dimension", "scale_band"],
        dropna=False,
    )
    .size()
    .rename("companies")
    .reset_index()
)
cluster_scale_band_mix["share_within_cluster_dimension"] = (
    cluster_scale_band_mix["companies"]
    / cluster_scale_band_mix.groupby(
        ["financial_cluster_id", "scale_dimension"]
    )["companies"].transform("sum")
)

display(
    cluster_sector_mix.sort_values(
        ["financial_cluster_id", "share_within_cluster"],
        ascending=[True, False],
    ).groupby("financial_cluster_id").head(5)
)

## 8. Associate historical period-t clusters with period-t+1 outcomes

This section reconstructs model features from each period-t row in table 03,
predicts its cluster using the fixed current-snapshot model, and joins the
period-t+1 transitions in table 04.

This is **retrospective external association analysis**, not production
forecast accuracy. A formal forecasting model must still use an
`available_date_t` time split and must be trained separately.

In [ ]:
CHANGE_BASE_COLUMNS = [
    ID,
    "period_t",
    "period_t_plus_1",
    "available_date_t",
    "available_date_t_plus_1",
    "gap_days",
]
CHANGE_T_FIELDS = [f"{field}_t" for field in RAW_BASE_FIELDS]
CHANGE_TARGET_FIELDS = [
    "current_assets_signed_log_change",
    "fixed_assets_signed_log_change",
    "creditors_total_signed_log_change",
    "equity_signed_log_change",
    "net_assets_liabilities_signed_log_change",
    "cash_signed_log_change",
    "debtors_signed_log_change",
    "employees_signed_log_change",
    "total_assets_less_current_liabilities_signed_log_change",
]

changes = pd.read_csv(
    input_paths["03_financial_change_labels.csv"],
    usecols=CHANGE_BASE_COLUMNS + CHANGE_T_FIELDS + CHANGE_TARGET_FIELDS,
    dtype={ID: "string"},
    low_memory=False,
)

transition_header = pd.read_csv(
    input_paths["04_financial_transition_labels.csv"],
    nrows=0,
).columns.tolist()
transition_target_columns = [
    column
    for column in transition_header
    if column.endswith(
        (
            "_onset_flag",
            "_recovery_flag",
            "_persistent_flag",
            "_onset_eligible",
            "_recovery_eligible",
            "_persistent_eligible",
        )
    )
]
transition_key_columns = [ID, "period_t", "period_t_plus_1"]
transitions = pd.read_csv(
    input_paths["04_financial_transition_labels.csv"],
    usecols=transition_key_columns + transition_target_columns,
    dtype={ID: "string"},
    low_memory=False,
)

pair_key = [ID, "period_t", "period_t_plus_1"]
assert changes.duplicated(pair_key).sum() == 0
assert transitions.duplicated(pair_key).sum() == 0

historical_pairs = changes[
    changes["gap_days"].between(MIN_GAP_DAYS, MAX_GAP_DAYS)
].copy()

historical_raw = historical_pairs[
    CHANGE_T_FIELDS
].rename(
    columns={f"{field}_t": field for field in RAW_BASE_FIELDS}
)
historical_engineered = engineer_features(historical_raw)
historical_pairs["model_feature_nonmissing_count"] = (
    historical_engineered.notna().sum(axis=1)
)

historical_eligible = historical_pairs[
    "model_feature_nonmissing_count"
].ge(6)
X_historical = transform_features(
    historical_engineered.loc[historical_eligible, MODEL_FEATURES]
)
historical_old_labels = final_model.predict(X_historical)
historical_new_labels = np.array(
    [old_to_new[int(label)] for label in historical_old_labels]
)

historical_clustered = historical_pairs.loc[
    historical_eligible
].copy()
historical_clustered["cluster_number"] = historical_new_labels
historical_clustered["financial_cluster_id"] = (
    historical_clustered["cluster_number"]
    .map(lambda value: f"FC{value:02d}")
)
historical_clustered["cluster_name_auto"] = (
    historical_clustered["financial_cluster_id"].map(name_map)
)

historical_clustered = historical_clustered.merge(
    transitions,
    on=pair_key,
    how="left",
    validate="one_to_one",
)

print(
    "Historical pairs:",
    len(changes),
    "| annual-gap eligible:",
    len(historical_pairs),
    "| clusterable:",
    len(historical_clustered),
)

In [ ]:
# Calculate each flag only when its matching eligibility field is True.
transition_rate_rows = []
for flag in [
    column
    for column in transition_target_columns
    if column.endswith("_flag")
]:
    eligible_column = flag.replace("_flag", "_eligible")
    if eligible_column not in historical_clustered.columns:
        continue

    eligible_rows = historical_clustered[
        historical_clustered[eligible_column].fillna(False)
    ]
    summary = (
        eligible_rows.groupby("financial_cluster_id")[flag]
        .agg(eligible_pairs="count", event_rate="mean")
        .reset_index()
    )
    summary["outcome"] = flag
    transition_rate_rows.append(summary)

cluster_future_transition_rates = pd.concat(
    transition_rate_rows,
    ignore_index=True,
)

cluster_future_change_summary = (
    historical_clustered.groupby("financial_cluster_id")[
        CHANGE_TARGET_FIELDS
    ]
    .agg(["count", "median", "mean"])
)
cluster_future_change_summary.columns = [
    f"{field}__{stat}"
    for field, stat in cluster_future_change_summary.columns
]
cluster_future_change_summary = (
    cluster_future_change_summary.reset_index()
)

display(
    cluster_future_transition_rates.pivot(
        index="financial_cluster_id",
        columns="outcome",
        values="event_rate",
    )
)

In [ ]:
if not cluster_future_transition_rates.empty:
    transition_plot_data = cluster_future_transition_rates[
        cluster_future_transition_rates["eligible_pairs"] >= 100
    ].copy()
    fig, ax = plt.subplots(figsize=(15, 7))
    sns.barplot(
        data=transition_plot_data,
        x="financial_cluster_id",
        y="event_rate",
        hue="outcome",
        ax=ax,
    )
    ax.set_title(
        "Subsequent financial transition rates by period-t cluster"
    )
    ax.set_ylabel("Event rate among eligible pairs")
    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0,
    )
    plt.tight_layout()
    transition_figure = (
        LOCAL_FIGURE_DIR / "cluster_future_transition_rates.png"
    )
    fig.savefig(transition_figure, dpi=180, bbox_inches="tight")
    plt.show()

## 9. Save results, model artefacts and the run manifest

The company-level output contains the current cluster, automatic profile name,
assignment confidence and data-quality controls. It is designed for direct use
in Tableau and the later financial-ranking module.

In [ ]:
output_tables = {
    "k_selection_metrics.csv": k_metrics_scored,
    "k_selection_shortlist.csv": shortlist,
    "company_cluster_assignments.csv": assignments,
    "cluster_profiles.csv": cluster_profiles,
    "cluster_sector_mix.csv": cluster_sector_mix,
    "cluster_account_category_mix.csv": cluster_account_category_mix,
    "cluster_scale_band_mix.csv": cluster_scale_band_mix,
    "cluster_future_transition_rates.csv": (
        cluster_future_transition_rates
    ),
    "cluster_future_change_summary.csv": (
        cluster_future_change_summary
    ),
}

for filename, frame in output_tables.items():
    path = LOCAL_RUN_OUTPUT_DIR / filename
    frame.to_csv(path, index=False)
    print("Saved", path, frame.shape)

model_bundle = {
    "run_id": RUN_ID,
    "selected_k": SELECTED_K,
    "automatic_recommended_k": AUTOMATIC_RECOMMENDED_K,
    "cluster_model": final_model,
    "old_to_new_cluster_mapping": old_to_new,
    "cluster_name_auto_mapping": name_map,
    "preprocessing": preprocessing_artifact,
}
model_path = LOCAL_RUN_MODEL_DIR / "financial_cluster_model.joblib"
joblib.dump(model_bundle, model_path)

experiment_config = {
    "run_id": RUN_ID,
    "aws_region": AWS_REGION,
    "s3_bucket": S3_BUCKET,
    "s3_input_prefix": S3_INPUT_PREFIX,
    "s3_output_run_prefix": S3_OUTPUT_RUN_PREFIX,
    "s3_model_run_prefix": S3_MODEL_RUN_PREFIX,
    "primary_cohort": PRIMARY_COHORT,
    "k_values": K_VALUES,
    "selected_k": SELECTED_K,
    "automatic_recommended_k": AUTOMATIC_RECOMMENDED_K,
    "selected_k_override": SELECTED_K_OVERRIDE,
        "selected_k_passes_all_rules": SELECTED_K_PASSES_ALL_RULES,
        "recommendation_status": RECOMMENDATION_STATUS,
        "maximum_cluster_share": MAX_CLUSTER_SHARE,
        "minimum_cluster_entropy": MIN_CLUSTER_ENTROPY,
        "minimum_stability_ari": MIN_STABILITY_ARI,
    "minimum_cluster_share": MIN_CLUSTER_SHARE,
    "random_state": RANDOM_STATE,
    "feature_groups": FEATURE_GROUPS,
    "quality_rule": preprocessing_artifact["quality_rule"],
    "historical_gap_days": [MIN_GAP_DAYS, MAX_GAP_DAYS],
    "notes": (
        "Clusters use latest current snapshot. Historical validation is "
        "retrospective association, not production forecast accuracy."
    ),
}
config_path = LOCAL_RUN_MODEL_DIR / "experiment_config.json"
config_path.write_text(
    json.dumps(experiment_config, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

mapping_path = LOCAL_RUN_MODEL_DIR / "cluster_name_mapping.json"
mapping_path.write_text(
    json.dumps(name_map, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved model:", model_path)
print("Saved config:", config_path)

In [ ]:
def upload_tree(local_dir: Path, s3_prefix: str) -> list[str]:
    uploaded = []
    for local_path in sorted(local_dir.rglob("*")):
        if not local_path.is_file():
            continue
        relative = local_path.relative_to(local_dir).as_posix()
        key = f"{s3_prefix}{relative}"
        s3.upload_file(str(local_path), S3_BUCKET, key)
        uploaded.append(f"s3://{S3_BUCKET}/{key}")
    return uploaded


uploaded_uris = []
if UPLOAD_TO_S3:
    uploaded_uris.extend(
        upload_tree(LOCAL_RUN_OUTPUT_DIR, S3_OUTPUT_RUN_PREFIX)
    )
    uploaded_uris.extend(
        upload_tree(LOCAL_RUN_MODEL_DIR, S3_MODEL_RUN_PREFIX)
    )

    if LOCAL_NOTEBOOK_PATH.exists():
        s3.upload_file(
            str(LOCAL_NOTEBOOK_PATH),
            S3_BUCKET,
            S3_NOTEBOOK_KEY,
        )
        uploaded_uris.append(
            f"s3://{S3_BUCKET}/{S3_NOTEBOOK_KEY}"
        )
    else:
        print(
            "Notebook file not found at expected path; "
            "results and models were still uploaded."
        )

upload_manifest = pd.DataFrame({"s3_uri": uploaded_uris})
display(upload_manifest)

## 10. Interpretation and next steps

The output is not a stand-alone cluster number. It is a structured set of
financial signals:

- `financial_cluster_id`: the company's current financial archetype;
- `cluster_name_auto`: a candidate human-readable descriptor;
- `assignment_margin`: how clearly the company belongs to its assigned cluster;
- `cluster_assignment_confidence`: High, Medium or Low;
- current reason codes for negative equity, working-capital deficit and creditor pressure;
- subsequent transition and financial-change distributions by period-t cluster.

BB / SME / Mid Corporate should later be analysed as
`business_segment × financial_cluster`, not added to the clustering features.
The ranking system can then combine the financial archetype with news and hiring
signals while keeping opportunity, momentum, risk and confidence as separate outputs.